# Data preprocessing - przygotowywanie danych do modelu

To see if virtual enviroment works properly, run this two commands below:

In [112]:
# import sys
# print(sys.executable)

### Import Python libraries
- numpy - library for working with arrays, also has functions for working in domain of linear algebra
- pandas - library for high-perfomance, easy-to-use data analysis and manipulation, providing data structures like DataFrames for tabular data
- scikit-learn (sklearn) - open source machine learning library that supports supervised and unsupervised learning, provides tools for model fitting, data preprocessing, model selection, evaluation, etc.

In [113]:
import numpy as np
import pandas as pd
import sklearn

print(f"sklearn version: {sklearn.__version__}")

sklearn version: 1.8.0


### Generate data

In [114]:
data = {
    'size': ['XL', 'L', 'M', 'L', 'M'],
    'color': ['red', 'blue', 'green', 'blue', 'red'],
    'gender': ['female', 'male', 'male', 'female', 'female'],
    'price': [199.0, 89.0, 99.0, 129.0, 79.0],
    'weight': [500, 450, 300, 380, 410],
    'bought': ['yes', 'no', 'yes', 'no', 'yes']
}

df_raw = pd.DataFrame(data=data)
df_raw

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,blue,male,89.0,450,no
2,M,green,male,99.0,300,yes
3,L,blue,female,129.0,380,no
4,M,red,female,79.0,410,yes


### Backup your data

In [115]:
df = df_raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   size    5 non-null      str    
 1   color   5 non-null      str    
 2   gender  5 non-null      str    
 3   price   5 non-null      float64
 4   weight  5 non-null      int64  
 5   bought  5 non-null      str    
dtypes: float64(1), int64(1), str(4)
memory usage: 372.0 bytes


### Changing data types and first exploration

In [116]:
for col in ['size', 'color', 'gender', 'bought']:
    df[col] = df[col].astype('category')

df['weight'] = df['weight'].astype('float')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   size    5 non-null      category
 1   color   5 non-null      category
 2   gender  5 non-null      category
 3   price   5 non-null      float64 
 4   weight  5 non-null      float64 
 5   bought  5 non-null      category
dtypes: category(4), float64(2)
memory usage: 420.0 bytes


describe() - shows only categories

In [117]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
price,5.0,119.0,48.476799,79.0,89.0,99.0,129.0,199.0
weight,5.0,408.0,75.299402,300.0,380.0,410.0,450.0,500.0


df.describe(include=['category']).T - shows statistics about our categories:
- count - counts rows of data
- unique - counts values wihout repetitions
- top - show the most repetetive value
- frequency - counts the occurrences of the most repetetive

In [118]:
df.describe(include=['category']).T

,count,unique,top,freq
size,5,3,L,2
color,5,3,blue,2
gender,5,2,female,3
bought,5,2,yes,3


## LabelEncoder

LabelEncoder is used to preproceed data in machine learning. It converts category variables into number labels. It assigns a unique integer (from 0 to ) to each category.

In [119]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(df['bought'])
le.transform(df['bought'])

array([1, 0, 1, 0, 1])

We can do it in one step like below:

In [120]:
le.fit_transform(df['bought'])

array([1, 0, 1, 0, 1])

le.classes_ is used to show mapping used in transforming.

In [121]:
le.classes_

array(['no', 'yes'], dtype=object)

In [122]:
df['bought'] = le.fit_transform(df['bought'])
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500.0,1
1,L,blue,male,89.0,450.0,0
2,M,green,male,99.0,300.0,1
3,L,blue,female,129.0,380.0,0
4,M,red,female,79.0,410.0,1


le.inverse_transform is used to inverse our transformation to previous state. It takes those numeric codes and converts them back to their original text labels.

In [123]:
df['bought'] = le.inverse_transform(df['bought'])
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500.0,yes
1,L,blue,male,89.0,450.0,no
2,M,green,male,99.0,300.0,yes
3,L,blue,female,129.0,380.0,no
4,M,red,female,79.0,410.0,yes


## OneHotEncoder

OneHotEncoder is another method for encoding categorical variables and works differently from LabelEncoder.

Instead of convering categories to single integers (like red=0, blue=1, green=2), OneHotEncoder creates three new columns:
- color_red: 1 if red, 0 otherwise,
- color_blue: 1 if blue, 0 otherwise
- color_green: 1 if green, 0 otherwise

In [124]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoder.fit(df[['size']])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_categ

In [125]:
encoder.transform(df[['size']])

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.]])

In [126]:
encoder.categories_

[array(['L', 'M', 'XL'], dtype=object)]

In [127]:
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoder.fit(df[['size']])
encoder.transform(df[['size']])

array([[0., 1.],
       [0., 0.],
       [1., 0.],
       [0., 0.],
       [1., 0.]])

In [128]:
encoder.categories_

[array(['L', 'M', 'XL'], dtype=object)]

In [129]:
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500.0,yes
1,L,blue,male,89.0,450.0,no
2,M,green,male,99.0,300.0,yes
3,L,blue,female,129.0,380.0,no
4,M,red,female,79.0,410.0,yes


## Pandas get_dummies()



In [130]:
pd.get_dummies(data=df)

,price,weight,size_L,size_M,size_XL,color_blue,color_green,color_red,gender_female,gender_male,bought_no,bought_yes
0,199.0,500.0,False,False,True,False,False,True,True,False,False,True
1,89.0,450.0,True,False,False,True,False,False,False,True,True,False
2,99.0,300.0,False,True,False,False,True,False,False,True,False,True
3,129.0,380.0,True,False,False,True,False,False,True,False,True,False
4,79.0,410.0,False,True,False,False,False,True,True,False,False,True


**pd.get_dummies(df, columns=['size'], drop_first=True)**
drop_first=True - filtered data, because we change categories into binaries and when we see bought 'yes'/'no' - we know if someone bought it or not, so two columns for this information is too much and unnecessary.

In [131]:
pd.get_dummies(df, columns=['size'], drop_first=True)

,color,gender,price,weight,bought,size_M,size_XL
0,red,female,199.0,500.0,yes,False,True
1,blue,male,89.0,450.0,no,False,False
2,green,male,99.0,300.0,yes,True,False
3,blue,female,129.0,380.0,no,False,False
4,red,female,79.0,410.0,yes,True,False


**pd.get_dummies(df, drop_first=True, prefix='new')** - we can set prefix for new columns created

In [132]:
pd.get_dummies(df, drop_first=True, prefix='new')

,price,weight,new_M,new_XL,new_green,new_red,new_male,new_yes
0,199.0,500.0,False,True,False,True,False,True
1,89.0,450.0,False,False,False,False,True,False
2,99.0,300.0,True,False,True,False,True,True
3,129.0,380.0,False,False,False,False,False,False
4,79.0,410.0,True,False,False,True,False,True


**pd.get_dummies(df, drop_first=True, prefix_sep='-')** - we can modify separator in created new columns


In [133]:
pd.get_dummies(df, drop_first=True, prefix_sep='-')

,price,weight,size-M,size-XL,color-green,color-red,gender-male,bought-yes
0,199.0,500.0,False,True,False,True,False,True
1,89.0,450.0,False,False,False,False,True,False
2,99.0,300.0,True,False,True,False,True,True
3,129.0,380.0,False,False,False,False,False,False
4,79.0,410.0,True,False,False,True,False,True


**pd.get_dummies(df, columns=['size'], drop_first=True)** - columns allow us to decide which column we want to convert

In [134]:
pd.get_dummies(df, columns=['size'], drop_first=True)

,color,gender,price,weight,bought,size_M,size_XL
0,red,female,199.0,500.0,yes,False,True
1,blue,male,89.0,450.0,no,False,False
2,green,male,99.0,300.0,yes,True,False
3,blue,female,129.0,380.0,no,False,False
4,red,female,79.0,410.0,yes,True,False


## Pandas get_dummies() vs OneHotEncoder

Both create the same one-hot encoded result, but they have important differences in when and how you should use them:
- return type:
  - get_dummies() - returns DataFrame with new column names like color_red
  - OneHotEncoder - returns NumPy array or sparse matrix without column names by default
- handling unknown categories:
  - get_dummies(): Creates columns only for categories it sees in that specific dataset. If new data has a new category, it silently ignores it
  - OneHotEncoder: Can be configured to handle unknown categories with handle_unknown='ignore' or raise an error, giving you more control
- training vs testing split:
  - get_dummies(): No memory of training data. If your test set is missing a category that was in training, you'll get different number of columns (causes errors!)
  - OneHotEncoder: Remembers all categories from .fit(), so .transform() on new data always produces the same columns



In [135]:
# Training data
train = pd.DataFrame({'color': ['red', 'blue', 'green']})
train_encoded = pd.get_dummies(train)  # 3 columns

# Test data (missing 'green')
test = pd.DataFrame({'color': ['red', 'blue']})
test_encoded = pd.get_dummies(test)  # Only 2 columns! - wrong!
pd.get_dummies(test)  # Only 2 columns! - wrong!


,color_blue,color_red
0,False,True
1,True,False


In [136]:
# With OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(train[['color']])
encoder.transform(test[['color']])  # Still 3 columns 

array([[0., 0., 1.],
       [1., 0., 0.]])

## Standarization - StandardScaler



In [137]:
print(f"{df['price']}\n")
print(f"average: {df['price'].mean()}")
print (f"standard deviation: {df['price'].std():.4f}")

0    199.0
1     89.0
2     99.0
3    129.0
4     79.0
Name: price, dtype: float64

average: 119.0
standard deviation: 48.4768


### Scaling data 3 methods:
1. Manual standarization
2. sklearn's scale()
3. StandardScaler (classic)

All three produce identical results mathematically, but StandardScaler is preferred because it solves the train/test problem:

**1. Manual standarization**
   
Raw formula - substract the mean and divide by standard deviation. Educational but not practical for production.

In [138]:
(df['price'] - df['price'].mean()) / df['price'].std()

0    1.650274
1   -0.618853
2   -0.412568
3    0.206284
4   -0.825137
Name: price, dtype: float64

**2. sklearn's scale**

Quick, stateless function that standardizes data in one call. It doesn't remember the mean/std, so it's only useful when you apply it to the same data

In [139]:
from sklearn.preprocessing import scale
scale(df['price'])

array([ 1.84506242, -0.69189841, -0.4612656 ,  0.2306328 , -0.92253121])

**3. StandardScaler**

The best practice approach. It's a class that remembers the mean and std from training data, can be reused on new/test data with the same parameters, integrates seamlessly into sklearn pipelines, can be saved and loaded for production models

In [140]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(df[['price']])
scaler.transform(df[['price']])

array([[ 1.84506242],
       [-0.69189841],
       [-0.4612656 ],
       [ 0.2306328 ],
       [-0.92253121]])

In [141]:
scaler = StandardScaler()
df[['price', 'weight']] = scaler.fit_transform(df[['price', 'weight']])
df

,size,color,gender,price,weight,bought
0,XL,red,female,1.845062,1.366002,yes
1,L,blue,male,-0.691898,0.623610,no
2,M,green,male,-0.461266,-1.603567,yes
3,L,blue,female,0.230633,-0.415740,no
4,M,red,female,-0.922531,0.029696,yes


## Preprocessing data to model - short summary

Generate data - copy from raw data from first step.

In [142]:
df = df_raw.copy()
df

,size,color,gender,price,weight,bought
0,XL,red,female,199.0,500,yes
1,L,blue,male,89.0,450,no
2,M,green,male,99.0,300,yes
3,L,blue,female,129.0,380,no
4,M,red,female,79.0,410,yes


Preprocessing data

In [143]:
le = LabelEncoder()

df['bought'] = le.fit_transform(df['bought'])

scaler = StandardScaler()
df[['price', 'weight']] = scaler.fit_transform(df[['price', 'weight']])

df = pd.get_dummies(df, drop_first=True)
df

,price,weight,bought,size_M,size_XL,color_green,color_red,gender_male
0,1.845062,1.366002,1,False,True,False,True,False
1,-0.691898,0.623610,0,False,False,False,False,True
2,-0.461266,-1.603567,1,True,False,True,False,True
3,0.230633,-0.415740,0,False,False,False,False,False
4,-0.922531,0.029696,1,True,False,False,True,False
